In [1]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd
import numpy as np

#Create temp directory
os.makedirs("temp", exist_ok=True)

# ---------- User-configurable paths ----------
path_data_intermediate = "/Users/nglei/Desktop/Academic/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate"   # e.g., "/Users/you/project/data/intermediate"
ccm_link_table_path   = os.path.join(path_data_intermediate, "CCMLinkingTable.csv")
annual_out_path  = os.path.join(path_data_intermediate, "a_aCompustat.parquet")
csv_export_path  = os.path.join(path_data_intermediate, "CompustatAnnual.csv")

#Connection to WRDS

require_wrds_load = False
if require_wrds_load == True:
    db = wrds.Connection(wrds_username="nglei2025")   # will ask for password if no .pgpass


In [6]:
#1 Load Compustat Fundamentals - All Variables

compustat_variables = [
    "gvkey","datadate","conm","fyear","tic","cusip","naicsh","sich",
    "aco","act","ajex","am","ao","ap","at","capx","ceq","ceqt","che","cogs",
    "csho","cshrc","dcpstk","dcvt","dlc","dlcch","dltis","dltr",
    "dltt","dm","dp","drc","drlt","dv","dvc","dvp","dvpa","dvpd",
    "dvpsx_c","dvt","ebit","ebitda","emp","epspi","epspx","fatb","fatl",
    "ffo","fincf","fopt","gdwl","gdwlia","gdwlip","gwo","ib","ibcom",
    "intan","invt","ivao","ivncf","ivst","lco","lct","lo","lt","mib",
    "msa","ni","nopi","oancf","ob","oiadp","oibdp","pi","ppenb","ppegt",
    "ppenls","ppent","prcc_c","prcc_f","prstkc","prstkcc","pstk","pstkl","pstkrv",
    "re","rect","recta","revt","sale","scstkc","seq","spi","sstk",
    "tstkp","txdb","txdi","txditc","txfo","txfed","txp","txt",
    "wcap","wcapch","xacc","xad","xint","xrd","xpp","xsga"
]

#Compustat variables full name
compustat_fullnames = """
    # identifiers / meta
    "a.gvkey": "Global Company Key (Compustat firm identifier)",
    "a.datadate": "Data Date (fiscal period end)",
    "a.conm": "Company Name",
    "a.fyear": "Fiscal Year",
    "a.tic": "Ticker Symbol",
    "a.cusip": "CUSIP (issue-level identifier)",
    "a.naicsh": "NAICS (industry code, historical)",
    "a.sich": "SIC Code – Primary",

    # balance sheet (currents & totals)
    "a.aco": "Current Assets – Other (Total)",
    "a.act": "Current Assets – Total",
    "a.ajex": "Adjustment Factor (Cumulative) for per-share items",
    "a.am": "Amortization of Intangibles (expense)",
    "a.ao": "Assets – Other (noncurrent)",
    "a.ap": "Accounts Payable – Trade",
    "a.at": "Assets – Total",

    # cash flow / investment / equity
    "a.capx": "Capital Expenditures",
    "a.ceq": "Common Equity – Total",
    "a.ceqt": "Common Equity – Total (Restated)",
    "a.che": "Cash and Short-Term Investments",

    # income statement (core)
    "a.cogs": "Cost of Goods Sold",
    "a.sale": "Sales/Revenue – Total",
    "a.revt": "Revenue – Total (same as Sales in most cases)",

    # shares / price
    "a.csho": "Common Shares Outstanding",
    "a.cshrc": "Common Shares Used to Calculate EPS (Basic)",
    "a.prcc_c": "Price – Close (fiscal year reference; Compustat ‘_c’ convention)",
    "a.prcc_f": "Price – Close (fiscal year end month)",

    # capital structure / debt
    "a.dcpstk": "Convertible Debt and Preferred Stock (combined legacy item)",
    "a.dcvt": "Debt – Convertible",
    "a.dlc": "Debt in Current Liabilities (Short-Term Debt)",
    "a.dlcch": "Changes in Current Debt (Cash Flow)",
    "a.dltis": "Long-Term Debt – Issuance (Cash Flow)",
    "a.dltr": "Long-Term Debt – Reduction (Cash Flow)",
    "a.dltt": "Long-Term Debt – Total",

    # D&A and related
    "a.dm": "Depreciation/Amortization – (legacy/misc. item; rarely used directly)",
    "a.dp": "Depreciation and Amortization (Income Statement)",

    # deferred revenue (current/LT captured elsewhere; you computed dr)
    "a.drc": "Deferred Revenue – Current",
    "a.drlt": "Deferred Revenue – Long-Term",

    # dividends (levels and per share)
    "a.dv": "Cash Dividends (Cash Flow)",
    "a.dvc": "Dividends – Common",
    "a.dvp": "Dividends – Preferred",
    "a.dvpa": "Dividends – Preferred (In Arrears)",
    "a.dvpd": "Dividends – Preferred (Declared/Payable)",
    "a.dvpsx_c": "Dividends per Share – by Ex-Date (Compustat convention ‘_c’)", 
    "a.dvt": "Dividends – Total",

    # profitability & ops
    "a.ebit": "Earnings Before Interest and Taxes (EBIT)",
    "a.ebitda": "Earnings Before Interest, Taxes, Depreciation & Amortization (EBITDA)",
    "a.emp": "Employees",
    "a.epspi": "Earnings per Share (Primary) – historical/legacy",
    "a.epspx": "Earnings per Share (Basic), excluding extraordinary items (legacy naming)",
    "a.ffo": "Funds From Operations",
    "a.fincf": "Financing Activities – Net Cash Flow",
    "a.fopt": "Financing Obligation – Present Value / Other Financing (rare)",
    "a.ib": "Income Before Extraordinary Items",
    "a.ibcom": "Income Before Extraordinary Items – Available for Common",
    "a.ni": "Net Income",
    "a.nopi": "Non-Operating Income",

    # intangibles / goodwill
    "a.gdwl": "Goodwill",
    "a.gdwlia": "Goodwill – Amortization",
    "a.gdwlip": "Goodwill – Impairment",
    "a.gwo": "Goodwill – Other",

    # intangibles & inventories
    "a.intan": "Intangible Assets (Total)",
    "a.invt": "Inventories – Total",

    # investments
    "a.ivao": "Investments and Advancements – Other",
    "a.ivncf": "Investing Activities – Net Cash Flow",
    "a.ivst": "Short-Term Investments",

    # current liabilities and other liabilities
    "a.lco": "Liabilities – Current – Other",
    "a.lct": "Current Liabilities – Total",
    "a.lo": "Liabilities – Other (noncurrent)",
    "a.lt": "Liabilities – Total",

    # minority / segments
    "a.mib": "Minority Interest (Balance Sheet)",
    "a.msa": "Marketable Securities – Adjustments / Misc. Segment Attribute (legacy)",

    # cash flow (operations)
    "a.oancf": "Operating Activities – Net Cash Flow",
    "a.ob": "Other Operating/Non-operating Balance (legacy)",
    "a.oiadp": "Operating Income After Depreciation",
    "a.oibdp": "Operating Income Before Depreciation (≈ EBITDA before non-operating items)",
    "a.pi": "Pretax Income",

    # PP&E
    "a.ppenb": "Property, Plant & Equipment – Net (Beginning balance)",
    "a.ppegt": "Property, Plant & Equipment – Gross Total",
    "a.ppenls": "Property, Plant & Equipment – Net (Disposals/Retirements)",
    "a.ppent": "Property, Plant & Equipment – Net (End of period)",

    # preferred stock
    "a.prstkc": "Treasury Stock (Common) – Dollar Amount",
    "a.prstkcc": "Treasury Stock (Common) – Contra/Other",
    "a.pstk": "Preferred Stock – Carrying Value (Total)",
    "a.pstkl": "Preferred Stock – Liquidating Value",
    "a.pstkrv": "Preferred Stock – Redemption Value",

    # retained earnings / receivables
    "a.re": "Retained Earnings",
    "a.rect": "Receivables – Total",
    "a.recta": "Receivables – Trade – Allowance (contra-asset)",

    # equity / stock transactions
    "a.scstkc": "Treasury Stock – Common (Shares) or Stock Common in Treasury (dollars) (legacy)",
    "a.seq": "Stockholders’ Equity – Total",
    "a.spi": "Special Items (after-tax)",
    "a.sstk": "Sale of Common and Preferred Stock (Cash Flow)",

    # treasury / taxes
    "a.tstkp": "Treasury Stock – Preferred (Dollar Amount)",
    "a.txdb": "Deferred Taxes – Balance Sheet",
    "a.txdi": "Deferred Taxes – Income (P&L)",
    "a.txditc": "Deferred Taxes & Investment Tax Credit (Balance Sheet)",
    "a.txfo": "Income Taxes – Foreign",
    "a.txfed": "Income Taxes – Federal",
    "a.txp": "Income Taxes – Provision (Current)",
    "a.txt": "Income Taxes – Total",

    # working capital and accruals
    "a.wcap": "Working Capital (Balance Sheet)",
    "a.wcapch": "Working Capital Change – Total (Statement of Changes/Cash Flow proxy)",
    "a.xacc": "Accrued Expenses (Liability) – Total",

    # expenses / R&D / interest / payables
    "a.xad": "Advertising Expense",
    "a.xint": "Interest and Related Expense – Total",
    "a.xrd": "Research and Development Expense",
    "a.xpp": "Prepaid Expenses (or Other Production Costs; legacy usage varies)",
    "a.xsga": "Selling, General & Administrative Expense",
"""

# Remove duplicates while preserving order
safe_compustat_variables = []
seen = set()
for var in compustat_variables:
    if var not in seen:
        safe_compustat_variables.append(var)
        seen.add(var)

print(f"Total unique variables: {len(safe_compustat_variables)}")

#Process all variables (Large output)
signals_to_process = [var for var in safe_compustat_variables
                     if var not in ["gvkey","datadate","conm","fyear",
                                    "tic","cusip","naicsh","sich",]]

print(f"Processing {len(signals_to_process)} variables: {signals_to_process}")


Total unique variables: 110


In [4]:
#2 Compustat All Variables Extraction From WRDS

if require_wrds_load == True:
    print("Loading Compustat data...")
    try:
        # Use raw SQL query to handle reserved words properly
        SQL = f"""
        SELECT
            {', '.join(safe_compustat_variables)}
        FROM comp.funda AS a
        WHERE a.consol = 'C'
          AND a.popsrc = 'D'
          AND a.datafmt = 'STD'
          AND a.curcd = 'USD'
          AND a.indfmt = 'INDL'
          AND a.datadate >= DATE '2000-01-01';
        """
        df = db.raw_sql(SQL, date_cols=['datadate']) # parse datadate as datetime
        print("Successfully loaded Compustat data using raw SQL")
    except Exception as e:
        raise RuntimeError(
            "WRDS pull failed. Ensure 'wrds' is installed and your credentials are set. "
            f"Original error: {e}"
        )

#Safety in pandas (after pull)
df = df[df['datadate'] >= pd.Timestamp('2000-01-01')]

# Export CSV for your IO-Momentum step
os.makedirs(path_data_intermediate, exist_ok=True)
df.to_csv(csv_export_path, index=False)


NameError: name 'require_wrds_load' is not defined

In [ ]:
#3 Cleaning Data

# ---------- 1) Require reasonable info (drop if at/prcc_c/ni missing) ----------
df = df.dropna(subset=['at', 'prcc_c', 'ni'])

# ---------- 2) 6-digit CUSIP ----------
df['cnum'] = df['cusip'].astype(str).str[:6]

# ---------- 3) Replace/derive variables (mirror Stata logic) ----------
# Deferred revenue: drc (current), drlt (LT)
df['dr'] = np.nan
cond_both = df['drc'].notna() & df['drlt'].notna()
cond_drc  = df['drc'].notna() & df['drlt'].isna()
cond_drlt = df['drc'].isna() & df['drlt'].notna()
df.loc[cond_both, 'dr']  = df.loc[cond_both, ['drc', 'drlt']].sum(axis=1)
df.loc[cond_drc,  'dr']  = df.loc[cond_drc,  'drc']
df.loc[cond_drlt, 'dr']  = df.loc[cond_drlt, 'drlt']

# Convertible debt and preferred stock:
# dcpstk = convertible debt + preferred stock (Compustat legacy), pstk = preferred stock, dcvt = convertible debt
df['dc'] = np.nan
cond1 = (df['dcpstk'] > df['pstk']) & df['pstk'].notna() & df['dcpstk'].notna() & df['dcvt'].isna()
df.loc[cond1, 'dc'] = df.loc[cond1, 'dcpstk'] - df.loc[cond1, 'pstk']
cond2 = df['pstk'].isna() & df['dcpstk'].notna() & df['dcvt'].isna()
df.loc[cond2, 'dc'] = df.loc[cond2, 'dcpstk']
cond3 = df['dc'].isna() & df['dcvt'].notna()
df.loc[cond3, 'dc'] = df.loc[cond3, 'dcvt']

# xint0 (interest expense total, 0 if missing)
df['xint0'] = df['xint'].fillna(0)

# xsga0 (SG&A, 0 if missing)
df['xsga0'] = df['xsga'].fillna(0)

# xad0 (advertising): 0 if missing (Stata: mi(xad) -> 0)
df['xad0'] = df['xad'].fillna(0)

# For these variables, missing → 0
zero_vars = [
    'nopi','dvt','ob','dm','dc','aco','ap','intan','ao','lco','lo','rect',
    'invt','drc','spi','gdwl','che','dp','act','lct','tstkp','dvpa','scstkc','sstk','mib',
    'ivao','prstkc','prstkcc','txditc','ivst'
]
for v in zero_vars:
    if v not in df.columns:
        df[v] = 0.0
    else:
        df[v] = df[v].fillna(0)

# ---------- 4) Join to CRSP–Compustat Linking Table and filter by link window ----------
# Expecting CCMLinkingTable.csv to contain at least:
# gvkey, permno, timeLinkStart_d, timeLinkEnd_d (dates)
ccm = pd.read_csv(ccm_link_table_path, parse_dates=['timeLinkStart_d', 'timeLinkEnd_d'])
# Stata's joinby gvkey unmatched(none) ≈ inner merge on gvkey
df = df.merge(ccm, on='gvkey', how='inner')

# Keep rows where datadate is within link validity: timeLinkStart_d <= datadate <= timeLinkEnd_d
# (Some links may have open-ended end dates; handle NaT as “no upper bound”)
left_ok  = df['timeLinkStart_d'].le(df['datadate'])
right_ok = df['timeLinkEnd_d'].isna() | df['datadate'].le(df['timeLinkEnd_d'])
df = df[left_ok & right_ok].copy()

# ---------- 5) Annual version ----------
# Drop link metadata you don’t want to carry forward
drop_cols = ['timeLinkStart_d', 'timeLinkEnd_d', 'linkprim', 'liid', 'linktype']
for col in drop_cols:
    if col in df.columns:
        df.drop(columns=col, inplace=True)

# Stata: destring gvkey (pandas can keep as string; if you need int, coerce here)
df['gvkey'] = pd.to_numeric(df['gvkey'], errors='coerce').astype('Int64')

# time_avail_m = month(datadate) + 6 (reporting lag of 6 months), stored as monthly period
datamonth = df['datadate'].dt.to_period('M')
df['time_avail_m'] = (datamonth + 6).dt.to_timestamp('MS')  # month start timestamp

# Save annual file
(df
 .sort_values(['gvkey', 'datadate'])
 .to_parquet(annual_out_path, index=False))